In [1]:
# Preprocess UKESM1 surface ozone data ~ needs 60GB memory
# Before adding a new scenario, check the data format

In [2]:
# WANT: Hourly surface ozone in ppb

In [3]:
import os
import numpy as np
import xarray as xr
from utils.utils import get_scenario_config, load_file_list, molmol_to_ppb, kgkg_to_ppb

In [4]:
# Load file and convert units
def load_ozone_ppb(ds, model_name):
    config = SCENARIO_CONFIG[model_name]
    var_name = config["var"]
    converter = config["convert"]

    data = ds[var_name]
    converted = converter(data)
    converted.attrs["units"] = "ppb"
    return converted


# === Return frequency string e.g '3-hourly' for xarray with time coord ===
def time_frequency(data):
    step = data.time.diff("time").median()
    # convert to hours
    step_hours = step / np.timedelta64(1, "h")
    # handle whole days if nicer
    if step_hours % 24 == 0:
        return f"{int(step_hours/24)}-daily"
    else:
        return f"{int(step_hours)}-hourly"


def interp_3hrly_to_1hrly(da):
    # Only do this if it's 3-hourly data
    if time_frequency(da) == "3-hourly":
        start_date = da["time"][0].item().strftime("%Y-%m-%d")
        end_date = da["time"][-1].item().strftime("%Y-%m-%d")
        calendar = da.time.encoding.get("calendar")

        # Build a full hourly time axis for the whole span
        hourly_time = xr.date_range(
            start_date + " 00:00:00",
            end_date + " 23:00:00",
            freq="h",
            calendar=calendar,
            use_cftime=True,
        )

        # Interpolate all at once
        da_interp = da.interp(time=hourly_time)

        del da, hourly_time

        return da_interp
    else:
        raise ValueError(f"Not 3-hourly data, write function to interpret {time_frequency(da)}")

In [5]:
# Map scenario -> variable name + conversion
# This may be different across scenarios for other models - CHECK
SCENARIO_CONFIG = {
    "SSP245_G6": {
        "var": "mass_fraction_of_ozone_in_air",
        "convert": kgkg_to_ppb},
    "G6-1.5K": {
        "var": "mass_fraction_of_ozone_in_air",
        "convert": kgkg_to_ppb},
    "hist": {
        "var": "sfo3",
        "convert": molmol_to_ppb},
}

In [6]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/file_paths/"
# Located in SCRATCH to save space
SAVE_DIR = f"/glade/derecho/scratch/awells/air_quality/{model}/ozone/hourly_o3/"

In [7]:
# === MAIN LOOP ===

for ens_num in [3]:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        da = load_ozone_ppb(xr.open_dataset(f), scenario)

        years = da["time"].dt.year
        # Perform this in 5 year batches to reduce memory load
        year_range_number = 5
        year_range = np.arange(
            years.min().item(), years.max().item() + 1,
            year_range_number)

        del da

        # Loop over year range and subset
        for start in year_range:
            end = start + (year_range_number - 1)
            da = load_ozone_ppb(xr.open_dataset(f), scenario)
            da_chunk = da.sel(time=(years >= start) & (years <= end))
            print(f"Years {start}-{end}, shape: {da_chunk.shape}")

            del da

            da_interp = interp_3hrly_to_1hrly(da_chunk)

            # Check format is hourly before saving
            if time_frequency(da_interp) != "1-hourly":
                raise ValueError("Input data is not hourly, "
                                 "convert data to hourly before continuing")

            # Create date stamp for file saving
            start_time = da_interp["time"][0].item().strftime("%Y%m%d")
            end_time = da_interp["time"][-1].item().strftime("%Y%m%d")
            dates = f"{start_time}-{end_time}"

            description = ("Processed hourly surface ozone "
                           "- scripts by A.F. Wells (2025)")
            da_interp.attrs["description"] = description
            da_interp.attrs["ensemble_number"] = ens_num
            da_interp.attrs["scenario"] = scenario
            da_interp.attrs["model"] = model

            out_file = f"Hourly_surface_o3_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            print(f"Saving to {out_path}")
            da_interp.to_netcdf(out_path)

            del da_interp

print("All processing complete.")

Processing SSP245_G6, ensemble 03
Reading ukesm_SSP245_G6_o3_3hr_03.nc
Years 2015-2019, shape: (14399, 144, 192)
Saving to /glade/derecho/scratch/awells/air_quality/UKESM1/ozone/hourly_o3/Hourly_surface_o3_UKESM1_SSP245_G6_03_20150101-20191230.nc
Years 2020-2024, shape: (14400, 144, 192)
Saving to /glade/derecho/scratch/awells/air_quality/UKESM1/ozone/hourly_o3/Hourly_surface_o3_UKESM1_SSP245_G6_03_20200101-20241230.nc
Years 2025-2029, shape: (14400, 144, 192)
Saving to /glade/derecho/scratch/awells/air_quality/UKESM1/ozone/hourly_o3/Hourly_surface_o3_UKESM1_SSP245_G6_03_20250101-20291230.nc
Years 2030-2034, shape: (14400, 144, 192)
Saving to /glade/derecho/scratch/awells/air_quality/UKESM1/ozone/hourly_o3/Hourly_surface_o3_UKESM1_SSP245_G6_03_20300101-20341230.nc
Years 2035-2039, shape: (14400, 144, 192)
Saving to /glade/derecho/scratch/awells/air_quality/UKESM1/ozone/hourly_o3/Hourly_surface_o3_UKESM1_SSP245_G6_03_20350101-20391230.nc
Years 2040-2044, shape: (14400, 144, 192)
Saving 